# 도구 호출(Tool Calling) + PostgreSQL 저장

`simple_tool.ipynb`에서는 도구가 `personal_info.txt`라는 **평문 텍스트 파일**에 이름·전화번호·주소를 한 줄씩 덧붙였습니다.
이 방식에는 몇 가지 문제가 있습니다.

- 여러 사용자가 동시에 저장하면 **쓰기가 섞이거나 깨질 수 있습니다** (파일 잠금이 없음).
- "이름으로 찾기", "오늘 들어온 기록만 보기" 같은 **조회가 불가능**합니다 (파일을 전부 읽어야 함).
- 접근 권한·백업·암호화 같은 **운영 관리가 전혀 안 됩니다**.

이 노트북은 **도구 호출 구조는 그대로 두고, 저장소만 PostgreSQL 테이블로 교체**합니다.
`record_personal_info_tool()` 내부가 `open(...).write(...)` 대신 `INSERT INTO ...`를 실행하는 것 말고는
LLM 쪽 코드(시스템 프롬프트, JSON 스키마, 도구 호출 루프)는 거의 바뀌지 않습니다.
**바뀌는 것은 "저장 방식"뿐, "도구 호출이라는 개념"은 그대로**라는 점을 눈여겨보세요.

## 사전 준비

1. PostgreSQL 서버가 실행 중이어야 합니다 (로컬 설치 또는 Docker).
   ```bash
   docker run --name local-postgres -e POSTGRES_PASSWORD=postgres -p 5432:5432 -d postgres
   ```
2. 드라이버 설치: `uv add "psycopg[binary]"` (이미 프로젝트에 추가되어 있습니다)
3. 연결 정보는 `.env`에 아래 변수로 넣어 두면 자동으로 읽습니다. 넣지 않으면 로컬 기본값을 씁니다.
   ```
   POSTGRES_HOST=localhost
   POSTGRES_PORT=5432
   POSTGRES_DB=postgres
   POSTGRES_USER=postgres
   POSTGRES_PASSWORD=postgres
   ```

> ⚠️ 실습에는 **가짜 개인정보**를 쓰세요. 실제 이름·전화번호·주소를 입력하면 DB에 그대로 저장됩니다.

## Step 1. 준비

In [2]:
import os
import json

import psycopg
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()

C:\Users\o3o_1\Desktop\Wins\lesson\ai\wins-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2. PostgreSQL 연결

연결 정보는 환경변수에서 읽습니다. `.env`에 값이 없으면 로컬 기본값(`localhost:5432`, 계정 `postgres`)을 씁니다.

`get_connection()`은 호출될 때마다 **새 연결을 만듭니다.** 전역 연결 하나를 계속 재사용하면
Gradio처럼 여러 요청이 겹칠 때 연결 상태가 꼬일 수 있기 때문입니다. 매번 열고 닫는 비용은
이 실습 규모에서는 무시할 만합니다.

In [3]:
DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": os.getenv("POSTGRES_PORT", "5432"),
    "dbname": os.getenv("POSTGRES_DB", "postgres"),
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "1234"),
}


def get_connection():
    return psycopg.connect(**DB_CONFIG)

## Step 3. 테이블 준비

`personal_info.txt` 한 줄(`이름\t전화번호\t주소`)에 해당하는 구조를 테이블 컬럼으로 그대로 옮깁니다.
`CREATE TABLE IF NOT EXISTS`라 여러 번 실행해도 안전합니다.

In [4]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS personal_info (
                id SERIAL PRIMARY KEY,
                name TEXT NOT NULL,
                phone TEXT NOT NULL,
                address TEXT NOT NULL,
                created_at TIMESTAMPTZ NOT NULL DEFAULT now()
            )
            """
        )

print("테이블 준비 완료")

테이블 준비 완료


## Step 4. 시스템 프롬프트

`simple_tool.ipynb`와 동일합니다. 저장소가 파일이든 DB든, **언제 도구를 불러야 하는지**를
LLM에게 알려주는 역할은 바뀌지 않습니다.

In [5]:
system_prompt = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

대화 중에 상담 기록을 남기기 위해 사용자의 이름, 전화번호, 주소를 물어보세요.
세 가지를 모두 받으면 record_personal_info_tool 도구를 호출해서 기록하세요.
세 가지가 다 모이기 전에는 도구를 호출하지 말고, 빠진 항목을 물어보세요.
"""

## Step 5. 도구로 쓸 파이썬 함수 — 파일 대신 DB에 저장

`simple_tool.ipynb`와 비교해 보세요. 바뀐 부분은 딱 한 군데, **저장 로직**뿐입니다.

| | `simple_tool.ipynb` | 이 노트북 |
|---|---|---|
| 저장 방식 | `open("personal_info.txt", "a")` 후 줄 추가 | `INSERT INTO personal_info (...)` |
| 함수 시그니처 | `(name, phone, address)` | 동일 |
| 반환값 | 문자열 (LLM에게 돌려줄 결과) | 동일 |

`with get_connection() as conn:` 블록을 예외 없이 벗어나면 psycopg가 **자동으로 커밋**합니다
(예외가 나면 자동 롤백). 파일의 `"a"` 모드처럼 명시적으로 플러시/클로즈를 챙길 필요가 없습니다.

In [6]:
def record_personal_info_tool(name, phone, address):
    print(f"Tool called to record: {name} / {phone} / {address}")
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO personal_info (name, phone, address) VALUES (%s, %s, %s)",
                (name, phone, address),
            )
    return "personal info is recorded"

In [15]:
record_personal_info_tool("홍길동", "(051) 777-9999", "부산시 해운대구")

Tool called to record: 홍길동 / (051) 777-9999 / 부산시 해운대구


'personal info is recorded'

## Step 6. LLM에게 도구를 설명하기 (JSON 스키마)

스키마는 **저장소와 무관**합니다. LLM은 파일에 쓰는지 DB에 쓰는지 알 필요가 없고,
"이 함수를 어떤 인자로 부르면 되는지"만 알면 됩니다. `simple_tool.ipynb`와 완전히 같습니다.

In [7]:
record_personal_info_tool_json = {
    "name": "record_personal_info_tool",
    "description": "Use this tool to record the name, phone number and address that a user provided",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The name of this user"},
            "phone": {"type": "string", "description": "The phone number of this user"},
            "address": {"type": "string", "description": "The address of this user"}
        },
        "required": ["name", "phone", "address"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": record_personal_info_tool_json}]

## Step 7. 도구 호출 처리하기

루프 구조(`finish_reason == "tool_calls"` 확인 → 인자 파싱 → 함수 실행 → 결과 붙여서 재호출)는
`simple_tool.ipynb`와 동일합니다. `record_personal_info_tool`만 DB 버전으로 바뀌었습니다.

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = record_personal_info_tool(
                args.get("name"), args.get("phone"), args.get("address")
            )
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})

        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content

## Step 8. 실행해 보기

1. "요즘 고민이 있어요" 처럼 아무 말이나 겁니다.
2. 모델이 이름·전화번호·주소를 물어봅니다.
3. **일부만** 알려 줍니다 → 도구를 부르지 않고 나머지를 물어보는지 확인하세요.
4. 나머지를 마저 알려 줍니다 → `Tool called to record: ...`가 찍히면 성공입니다.
5. 다음 Step에서 DB에 실제로 들어갔는지 SQL로 조회해 확인합니다.

> 다시 말씀드리지만 **가짜 정보**를 쓰세요.

In [9]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 9. 저장된 데이터 조회하기

파일 저장 방식(`simple_tool.ipynb`)은 "파일을 열어서 눈으로 확인"하는 것이 전부였습니다.
DB로 바꾸면 **SQL로 바로 조회**할 수 있다는 것이 가장 큰 실용적 차이입니다.
텍스트 파일로는 "이름으로 검색", "최근 N건만 보기" 같은 질의가 번거롭지만, 여기서는 한 줄이면 됩니다.

In [10]:
def fetch_all_records():
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT id, name, phone, address, created_at FROM personal_info ORDER BY id"
            )
            return cur.fetchall()


for row in fetch_all_records():
    print(row)

(1, '홍길동', '010 1234 5678', '부산광역시 해운대구 센텀시티', datetime.datetime(2026, 10, 15, 13, 12, 17, 379070, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))


## Step 10. `@function_tool`로 만들기 (Agents SDK, DB 버전)

`simple_tool.ipynb` Step 8과 같은 비교입니다. 함수의 타입 힌트·독스트링에서 스키마가 자동으로 만들어지고,
도구 호출 루프(`finish_reason` 확인 → 파싱 → 실행 → 재호출)를 SDK가 대신 처리합니다.
**바뀐 것은 여기서도 함수 내부의 저장 로직뿐입니다.**

In [11]:
from agents import Agent, Runner, function_tool


@function_tool
def record_personal_info_tool_v2(name: str, phone: str, address: str) -> str:
    """
    사용자가 알려 준 이름, 전화번호, 주소를 PostgreSQL에 기록한다.

    Args:
        name: 사용자 이름
        phone: 사용자 전화번호
        address: 사용자 주소
    """
    print(f"Tool called to record: {name} / {phone} / {address}")
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO personal_info (name, phone, address) VALUES (%s, %s, %s)",
                (name, phone, address),
            )
    return "personal info is recorded"

In [12]:
agent = Agent(
    name="상담 도우미",
    instructions=system_prompt,
    model="gpt-5.4-mini",
    tools=[record_personal_info_tool_v2],
)

# 정보가 부족할 때 — 도구를 부르지 않고 물어봐야 한다
result = await Runner.run(agent, "요즘 고민이 많아요. 제 이름은 김철수입니다.")
print(result.final_output)

많이 힘드셨겠어요. 고민, 편하게 말씀해 주세요.  
상담 기록을 위해 **전화번호와 주소**도 알려주실 수 있을까요?


In [13]:
# 세 가지를 모두 줄 때 — 도구를 부른다
result = await Runner.run(agent, "이름 김철수, 전화 010-1234-5678, 주소 서울시 강남구입니다.")
print(result.final_output)

Tool called to record: 김철수 / 010-1234-5678 / 서울시 강남구입니다.
기록해두었습니다.


### Gradio에 붙이기

In [14]:
def to_agent_input(history, message):
    """gradio 기록을 에이전트 입력으로 바꾼다. role/content만 남긴다."""
    items = []
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")
        if role in ("user", "assistant") and isinstance(content, str):
            items.append({"role": role, "content": content})
    items.append({"role": "user", "content": message})
    return items


async def chat_with_agent(message, history):
    result = await Runner.run(agent, to_agent_input(history, message))
    return result.final_output


gr.ChatInterface(chat_with_agent).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [16]:
gr.close_all()

Closing server running on port: 7860
Closing server running on port: 7861


## 정리

| | 파일(`personal_info.txt`) | PostgreSQL |
|---|---|---|
| 저장 코드 | `open(...).write(...)` | `INSERT INTO ...` |
| 동시 쓰기 | 여러 요청이 겹치면 깨질 수 있음 | 트랜잭션으로 안전하게 처리됨 |
| 조회 | 파일 전체를 읽어서 눈으로 확인 | `SELECT ... WHERE ...`로 즉시 검색 |
| 구조 검증 | 없음 (아무 줄이나 추가됨) | 컬럼 타입·제약조건(`NOT NULL` 등)으로 강제 |
| 접근 제어·백업 | 파일 시스템 권한에 의존 | DB 계정·권한, 백업 도구 사용 가능 |
| 확장 | 파일이 커지면 전체를 다시 읽어야 함 | 인덱스로 대량 데이터도 빠르게 조회 |

**기억할 것**

- 도구 호출의 본질(LLM은 "무엇을 부를지"만 정하고, 실행은 우리 코드가 한다)은 저장소를 무엇으로 바꾸든 그대로입니다.
- 저장소를 바꾸는 작업은 **도구 함수 내부 구현**에만 영향을 주고, 시스템 프롬프트·JSON 스키마·도구 호출 루프는 건드릴 필요가 없습니다.
- 실제 서비스에서 개인정보를 다룬다면 평문 텍스트 파일보다 접근 제어·감사 로그·백업이 가능한 DB가 훨씬 적합합니다.

## 스스로 해 보기

1. `personal_info` 테이블에 `UNIQUE` 제약(예: 전화번호)을 추가하고, 중복 저장 시 어떤 오류가 나는지 확인해 보세요.
2. 이름으로 기존 기록을 검색해서 "이미 상담하신 적이 있네요"라고 답하는 두 번째 도구를 만들어 보세요.
3. `record_personal_info_tool`이 DB 연결에 실패하면 어떻게 될까요? `try/except`로 감싸고 실패 메시지를 LLM에게 돌려주도록 고쳐 보세요.
4. `../day2/security_rag_advance`는 구조화된 사실(위험도, 버전 판정)을 DB가 아니라 JSON 파일에서 읽습니다. 이 노트북처럼 PostgreSQL로 옮긴다면 무엇이 좋아질까요?